## Display existing sim data

In [ ]:
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.read_csv("aggregate_results.csv")
df = df.rename(columns={'periodA': 'period'})

inputs = ["period", "nodes", "pktsizeB"]
outputs = ["sentA", "recvA", "lossA", "sentB", "recvB", "lossB"]

periods = df['period'].unique()
df['nodes'] = df['nodesA'].astype("string") + ", " + df['nodesB'].astype("string")
df = df.drop(columns=['Unnamed: 0', 'periodB'])
nodes = df['nodes'].unique()
pktsizeBs = df['pktsizeB'].unique()
datamodes=df['dataMode'].unique()

w_periods = widgets.Dropdown(options=periods,
                 description="Period (s): ",
                 layout={'width': 'max-content'},
                 style={'description_width': '150px'},
                 disabled=False)
w_nodes = widgets.Dropdown(options=nodes,
                 description="Node counts (A, B): ",
                 layout={'width': 'max-content'},
                 style={'description_width': '150px'},
                 disabled=False)
w_datamode = widgets.Dropdown(options=datamodes,
                 description="Data Mode: ",
                 layout={'width': 'max-content'},
                 style={'description_width': '150px'},
                 disabled=False)
w_pktsizeB = widgets.Dropdown(options=pktsizeBs,
                 description="B Payload Size (bytes): ",
                 layout={'width': 'max-content'},
                 style={'description_width': '150px'},
                 disabled=False)
w_selection = widgets.Dropdown(options=inputs,
                 description="X-axis: ",
                 layout={'width': 'max-content'},
                 style={'description_width': '150px'},
                 disabled=False)
w_help1 = widgets.Label("Note: data mode numbers correspond to US regional standard:")
w_help2 = widgets.Label("0=sf 10, 1=sf 9, 2=sf 8, 3=sf 7, -1=selected by sim (optimal for range)")
w_help3 = widgets.Label("Group A fixed to 25 Byte payload, Group B is variable size burst of max-size packets until payload size")

w_print = widgets.Checkbox(
    value=False,
    description='Print Data',
    disabled=False,
    indent=False
)
w_axis = widgets.Checkbox(
    value=True,
    description='Fixed y-axis',
    disabled=False,
    indent=False
)

def run_func (period_val, node_val, datamode_val, pktsizeB_val, selection_val, fix_axis, do_print):
    graphtext = ""
    df_selection = None
    if selection_val == "period":
        df_selection = df.query(f"nodes == \"{node_val}\" and dataMode == {datamode_val} and pktsizeB == {pktsizeB_val}")
        graphtext = graphtext + f"Nodes (A, B)={node_val}\nData Mode={datamode_val}\nB Payload Size={pktsizeB_val}"
    elif selection_val == "nodes":
        df_selection = df.query(f"period == {period_val} and dataMode == {datamode_val} and pktsizeB == {pktsizeB_val}")
        graphtext = graphtext + f"Tx event period={period_val}\nData Mode={datamode_val}\nB Payload Size={pktsizeB_val}"
    elif selection_val == "pktsizeB":
        df_selection = df.query(f"period == {period_val} and nodes == \"{node_val}\" and dataMode == {datamode_val}")
        graphtext = graphtext + f"Tx event period={period_val}\nNodes (A, B)={node_val}\nData Mode={datamode_val}\n"
    else:
        print("Error: Invalid selection")
        return

    xscale = selection_val
    if selection_val == 'nodes':
        xscale = 'nodesA'

    plt.figure(figsize=(10,6))
    plt.plot(df_selection[xscale].astype("int"), df_selection['lossA'],marker='o')
    plt.plot(df_selection[xscale].astype("int"), df_selection['lossB'],marker='o')
    plt.legend(["Group A", "Group B"])
    plt.xlabel(xscale)
    plt.ylabel("Packet loss %")
    plt.xlim((max(df_selection[xscale]) * -0.1, max(df_selection[xscale])*1.1))
    if fix_axis:
        plt.ylim((-5, 100))
    plt.title(f"Packet Loss over {selection_val}")
    plt.figtext(0.15, 0.75, graphtext)
    plt.show()

    print(f"Period: {period_val}, Nodes (A, B): ({node_val}), B Payload Size: {pktsizeB_val}, Data Mode: {datamode_val}")
    if do_print:
        print(df_selection)

inputWidgets = widgets.VBox([w_periods, w_nodes, w_pktsizeB, w_selection, w_datamode, w_help1, w_help2, w_help3, w_axis, w_print])

plotter = widgets.interactive_output (run_func,
                                      {"period_val": w_periods,
                                       "node_val": w_nodes,
                                       "datamode_val": w_datamode,
                                       "pktsizeB_val": w_pktsizeB,
                                       "selection_val": w_selection,
                                       "fix_axis": w_axis,
                                       "do_print": w_print})



display(inputWidgets, plotter)

## Run simulation with parameters:

In [ ]:
### Note: The structure of this file was initially generated with Gemini
### and I filled in the specific values and command text.

import ipywidgets as widgets
from IPython.display import display, clear_output
import subprocess

# Define the input widgets
period_a = widgets.IntText(value=600,
                           style={'description_width': '150px'},
                           description='Period A (s):')
period_b = widgets.IntText(value=600,
                           style={'description_width': '150px'},
                           description='Period B (s):')
num_nodes_a = widgets.IntText(value=20,
                              style={'description_width': '150px'},
                              description='Nodes A:')
num_nodes_b = widgets.IntText(value=5,
                              style={'description_width': '150px'},
                              description='Nodes B:')
packet_size_a = widgets.IntText(value=25,
                                style={'description_width': '150px'},
                                description='Payload Size A:')
packet_size_b = widgets.IntText(value=500,
                                style={'description_width': '150px'},
                                description='Payload Size B:')
data_mode = widgets.IntText(value=-1,
                            style={'description_width': '150px'},
                            description='Data Mode:')
seed = widgets.IntText(value=3,
                       style={'description_width': '150px'},
                       description='Seed:')

dm_help = widgets.Label("Note: data mode numbers correspond to US regional standard. 0=sf10, 1=sf9, 2=sf8, 3=sf7, -1=optimal sf")

# Create a button to trigger the execution

try:
    run_button.close()
except NameError:
    pass

run_button = widgets.Button(
    description="Run Program",
    button_style='success', # Gives the button a green color
    icon='play'
)

# Create an output area to capture printed text and stdout
output_area = widgets.Output()

# Define the function that runs when the button is clicked
def on_run_button_clicked(b):
    with output_area:
        # Clear the previous output
        clear_output()

        # Extract integer values from the widgets
        tx_period_a = period_a.value
        tx_period_b = period_b.value
        nodes_a = num_nodes_a.value
        nodes_b = num_nodes_b.value
        ps_a = packet_size_a.value
        ps_b = packet_size_b.value
        dm = data_mode.value
        s = seed.value

        # Specify the path to your executable
        program_name = "../../ns3"

        # Construct the command line argument list.
        command = f"../../ns3 run --no-build \"wes-simulation --appPeriodA={tx_period_a} --appPeriodB={tx_period_b}" \
                    f" --nDevicesA={nodes_a} --nDevicesB={nodes_b}"\
                    f" --packetSizeA={ps_a} --packetSizeB={ps_b} --dataMode={dm} --seed={s}\""

        print(f"Executing command: {command}\n")

        try:
            # Run the process, capture stdout and stderr as text
            result = subprocess.run(command,capture_output=True,text=True,shell=True)
            # print("periodA, nodesA, sentA, recvA, lossA, periodB, nodesB, sentB, recvB, lossB, dataMode, seed, pktsizeB")
            # print(result.stdout)
            resultstr = result.stdout.split(",")
            resultstr = [result.strip() for result in resultstr]
            print(f"A Sent / Recv / Packet loss %")
            print(f"{resultstr[2]} / {resultstr[3]} / {float(resultstr[4]):.2f}%\n")
            print(f"B Sent / Recv / Packet loss %")
            print(f"{resultstr[7]} / {resultstr[8]} / {float(resultstr[9]):.2f}%\n")

        except subprocess.CalledProcessError as e:
            print("--- Program Execution Failed ---")
            print(f"Return code: {e.returncode}")
            print(f"Standard Error:\n{e.stderr}")
        except FileNotFoundError:
            print(f"Error: Program '{program_name}' could not be found.")
            print("Please update the 'program_name' variable in the code to point to your actual executable.")

# Link the button click event to the function
run_button.on_click(on_run_button_clicked)

# Group the widgets visually and display them
input_widgets = widgets.VBox([
    period_a, period_b, num_nodes_a, num_nodes_b,
    packet_size_a, packet_size_b, data_mode, seed, dm_help
])

display(input_widgets, run_button, output_area)
